---
toc: true
image: example.png
pub-info:
    abstract: |
        A simulation that starts empty is not yet representative of steady-state
        behaviour, so it's standard practice to discard an initial warm-up period from
        reported results. Filtering the event log by time to do this quietly breaks an
        animation, though - it deletes the arrival record of everyone already in the
        system, so they vanish from every frame, including ones still queuing. This
        walks through `reshape_for_animations`/`animate_activity_log`'s `warm_up=` and
        `snapshot_alignment=` parameters, which trim the animation window instead of
        the log, so that history stays intact.
execute:
  enabled: true
---

# Feature Example: Discarding a Warm-up Period from an Animation

`event_durations`, `queue_size_over_time` and the rest of `vidigi.analysis` all
take a `warm_up=` parameter for excluding early, unrepresentative results - see
[feat_warm_up.ipynb](../feat_warm_up/feat_warm_up.ipynb) for choosing how much. This
notebook is about a related but distinct problem: applying that same warm-up to an
*animation*, where the obvious approach - filtering the event log by time - does not
just exclude early data, it actively corrupts the result.

## Model setup

The same single-step clinic model used by
[example_1_simplest_case](../example_1_simplest_case/ex_1_simplest_case.ipynb) - one
queue, one resource, patients arrive, wait for a treatment cubicle, are treated, and
leave.

In [ ]:
from feat_animation_warm_up_model_classes import Trial, g
from vidigi.prep import reshape_for_animations
from vidigi.animation import animate_activity_log
from vidigi.utils import EventPosition, create_event_position_df
import pandas as pd
import os

import plotly.io as pio
pio.renderers.default = "notebook"

In [ ]:
#| echo: false
#| output: asis
file_path = "feat_animation_warm_up_model_classes.py"

with open(file_path, "r") as f:
    code_content = f.read()

print(f"""
:::{{.callout-note collapse="true"}}
### View Imported Code, which has had logging steps added at the appropriate points in the 'model' class

```python
{code_content}
```

:::

""")

In [ ]:
g.number_of_runs = 1

my_trial = Trial()
my_trial.run_trial()

event_log = my_trial.all_event_logs[my_trial.all_event_logs["run"] == 0].reset_index(drop=True)
event_log.head()

4 cubicles, a fairly busy arrival rate, and a 600 time-unit run - a queue is
already well established by the time we cut it off at `warm_up = 100`.

In [ ]:
warm_up = 100

still_present_at_warm_up = (
    event_log[event_log["event"] == "arrival"][["patient", "time"]]
    .rename(columns={"time": "arrival_time"})
    .merge(
        event_log[event_log["event"] == "depart"][["patient", "time"]]
        .rename(columns={"time": "depart_time"}),
        on="patient", how="left",
    )
)
still_present_at_warm_up = still_present_at_warm_up[
    (still_present_at_warm_up["arrival_time"] < warm_up)
    & (still_present_at_warm_up["depart_time"].isna() | (still_present_at_warm_up["depart_time"] >= warm_up))
]
still_present_at_warm_up

These are the patients genuinely still in the system - mid-queue or mid-treatment -
at the moment `warm_up` ends. Any correct animation of the post-warm-up period has to
show them.

## The problem: filtering the log breaks the animation

The obvious way to discard a warm-up period is `event_log[event_log["time"] >= warm_up]`.
`reshape_for_animations` warns immediately if you try it:

In [ ]:
naive_filtered_log = event_log[event_log["time"] >= warm_up]

naive_reshaped = reshape_for_animations(
    naive_filtered_log, every_x_time_units=10, limit_duration=g.sim_duration,
    entity_col_name="patient", run_col_name="run",
)

The warning names exactly why: filtering removed the `arrival` row of every
patient who arrived *before* `warm_up` but is still in the system - `reshape_for_animations`
works out who is present at each snapshot from the arrival and departure rows, so an
entity with no `arrival` row is never drawn, in *any* frame, not just the first one:

In [ ]:
present_anywhere = set(naive_reshaped["patient"].dropna().unique())
missing = set(still_present_at_warm_up["patient"]) - present_anywhere
f"{len(missing)} of {len(still_present_at_warm_up)} genuinely-present patients never appear anywhere in the naively-filtered animation"

## The fix: `warm_up=`

Pass the **whole** event log instead, and let `warm_up=` trim the animation window -
the history before it is kept around just long enough to work out who is present when
the window opens:

In [ ]:
proper_reshaped = reshape_for_animations(
    event_log, every_x_time_units=10, limit_duration=g.sim_duration,
    entity_col_name="patient", run_col_name="run", warm_up=warm_up,
)

first_frame = proper_reshaped[proper_reshaped["snapshot_time"] == warm_up]
sorted(first_frame["patient"].dropna().unique())

Every one of the patients identified above is there, in the very first frame -
no warning, nothing missing.

`animate_activity_log` takes the same `warm_up=` and forwards it straight through, so
the fix is identical whichever level you're working at:

In [ ]:
event_position_df = create_event_position_df([
    EventPosition(event="arrival", x=50, y=300, label="Arrival"),
    EventPosition(event="treatment_wait_begins", x=205, y=275, label="Waiting for Treatment"),
    EventPosition(event="treatment_begins", x=205, y=175, resource="n_cubicles", label="Being Treated"),
    EventPosition(event="depart", x=270, y=70, label="Exit"),
])

fig = animate_activity_log(
    event_log=event_log,
    event_position_df=event_position_df,
    entity_col_name="patient",
    scenario=g(),
    setup_mode=False,
    every_x_time_units=10,
    warm_up=warm_up,
    include_play_button=True,
    resource_icon_size=15,
    text_size=20,
    entity_icon_size=13,
    gap_between_entities=6,
    gap_between_queue_rows=25,
    gap_between_resource_rows=25,
    plotly_height=600,
    frame_duration=200,
    plotly_width=1000,
    override_x_max=300,
    override_y_max=500,
    limit_duration=g.sim_duration,
    wrap_queues_at=25,
    step_snapshot_max=125,
    time_display_units="dhm",
    display_stage_labels=False,
    add_background_image="https://raw.githubusercontent.com/Bergam0t/vidigi/refs/heads/main/examples/example_1_simplest_case/Simplest%20Model%20Background%20Image%20-%20Horizontal%20Layout.drawio.png",
)
fig

The animation opens already populated - a queue and a full set of cubicles, not
the empty system the model actually started in - and runs from `t=100` to the end of
the run.

## `snapshot_alignment`: choosing where the grid starts

`warm_up=100` and `every_x_time_units=10` divide evenly, so there's only one sensible
snapshot grid. That's not always true - with `every_x_time_units=15`, `100` isn't a
multiple of `15`, and `snapshot_alignment` decides what happens:

In [ ]:
from_warm_up = reshape_for_animations(
    event_log, every_x_time_units=15, limit_duration=g.sim_duration,
    entity_col_name="patient", run_col_name="run", warm_up=warm_up,
    snapshot_alignment="warm_up",
)
from_run_start = reshape_for_animations(
    event_log, every_x_time_units=15, limit_duration=g.sim_duration,
    entity_col_name="patient", run_col_name="run", warm_up=warm_up,
    snapshot_alignment="run_start",
)

sorted(from_warm_up["snapshot_time"].unique())[:6], sorted(from_run_start["snapshot_time"].unique())[:6]

`snapshot_alignment="warm_up"` (the default) puts the first frame exactly on the
boundary, `100`, so the animation opens showing the system precisely as the warm-up
ends. `snapshot_alignment="run_start"` keeps the grid that would have applied with no
warm-up at all - multiples of `15` from `0` - and simply drops the frames before
`100`, so the first surviving one is `105`. Frame times then stay the same round
numbers a no-warm-up run would have used, at the cost of the first frame no longer
landing exactly on the cutoff. The two are identical whenever `warm_up` happens to be
a multiple of `every_x_time_units`, as it was above.

## Closing note

This and [feat_warm_up.ipynb](../feat_warm_up/feat_warm_up.ipynb) answer two different
questions about the same idea. That notebook is about *choosing* a warm-up length for
reported statistics, using `plot_warm_up_diagnostic`/`welch_moving_average` on a
duration or queue-length series. This one assumes a length has already been chosen,
and is about *applying* it to an animation without silently losing entities in the
process - a mistake the new warning above exists specifically to catch.

[examples/v2_release_additions.ipynb](../v2_release_additions/v2_release_additions.ipynb)
tours the rest of what shipped alongside this in 2.0.0.